# 02 — Exploratory Data Analysis
Goal: identify where the strongest ADHD predictive signal lives — connectome, metadata, or both.
`Sex_F` is loaded only to be excluded later; it is never used as a feature or plotted here to avoid building intuition around it as a predictor.

In [1]:

import os
os.chdir('/home/claude/adhd_project')
import numpy as np
import pandas as pd
from sklearn.feature_selection import mutual_info_classif, f_classif
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

BASE = 'data/raw/widsdatathon2025'
train_cat = pd.read_excel(f'{BASE}/TRAIN_NEW/TRAIN_CATEGORICAL_METADATA_new.xlsx')
train_quan = pd.read_excel(f'{BASE}/TRAIN_NEW/TRAIN_QUANTITATIVE_METADATA_new.xlsx')
train_sol = pd.read_excel(f'{BASE}/TRAIN_NEW/TRAINING_SOLUTIONS.xlsx')
train_fcm = pd.read_csv(f'{BASE}/TRAIN_NEW/TRAIN_FUNCTIONAL_CONNECTOME_MATRICES_new_36P_Pearson.csv')

y = train_sol.set_index('participant_id')['ADHD_Outcome']
print('Loaded. y shape:', y.shape)


Loaded. y shape: (1213,)


## ADHD target class balance

In [2]:

print(y.value_counts())
print((y.value_counts(normalize=True)*100).round(2))
print('Imbalance ratio: %.2f : 1 (ADHD-positive : ADHD-negative)' % (y.value_counts()[1]/y.value_counts()[0]))


ADHD_Outcome
1    831
0    382
Name: count, dtype: int64
ADHD_Outcome
1    68.51
0    31.49
Name: proportion, dtype: float64
Imbalance ratio: 2.18 : 1 (ADHD-positive : ADHD-negative)


## Metadata: missingness, variance, and correlation with ADHD_Outcome
Quantitative columns only (categorical handled separately). Age has ~30% missingness — check whether that's meaningful before deciding to drop it.

In [3]:

quan_cols = [c for c in train_quan.columns if c != 'participant_id']
quan_full = train_quan.merge(train_sol[['participant_id','ADHD_Outcome']], on='participant_id')

miss_pct = (train_quan[quan_cols].isnull().mean()*100).round(2).sort_values(ascending=False)
print('Missingness % by quantitative column:')
print(miss_pct)


Missingness % by quantitative column:
MRI_Track_Age_at_Scan         29.68
ColorVision_CV_Score           1.90
EHQ_EHQ_Total                  1.07
APQ_P_APQ_P_CP                 0.99
APQ_P_APQ_P_INV                0.99
APQ_P_APQ_P_ID                 0.99
APQ_P_APQ_P_PM                 0.99
APQ_P_APQ_P_PP                 0.99
APQ_P_APQ_P_OPD                0.99
SDQ_SDQ_Conduct_Problems       0.74
SDQ_SDQ_Difficulties_Total     0.74
SDQ_SDQ_Emotional_Problems     0.74
SDQ_SDQ_Generating_Impact      0.74
SDQ_SDQ_Externalizing          0.74
SDQ_SDQ_Hyperactivity          0.74
SDQ_SDQ_Internalizing          0.74
SDQ_SDQ_Peer_Problems          0.74
SDQ_SDQ_Prosocial              0.74
dtype: float64


In [4]:

# Point-biserial-style correlation (Spearman) between each quantitative feature and ADHD_Outcome
corrs = {}
for c in quan_cols:
    sub = quan_full[[c, 'ADHD_Outcome']].dropna()
    corrs[c] = sub[c].corr(sub['ADHD_Outcome'], method='spearman')
corr_series = pd.Series(corrs).sort_values(key=abs, ascending=False)
print('Spearman correlation of each quantitative feature with ADHD_Outcome:')
print(corr_series.round(3))


Spearman correlation of each quantitative feature with ADHD_Outcome:
SDQ_SDQ_Hyperactivity         0.548
SDQ_SDQ_Externalizing         0.519
SDQ_SDQ_Difficulties_Total    0.476
SDQ_SDQ_Generating_Impact     0.417
SDQ_SDQ_Conduct_Problems      0.283
SDQ_SDQ_Internalizing         0.275
SDQ_SDQ_Peer_Problems         0.229
SDQ_SDQ_Emotional_Problems    0.217
SDQ_SDQ_Prosocial            -0.198
APQ_P_APQ_P_OPD               0.143
APQ_P_APQ_P_ID                0.116
APQ_P_APQ_P_INV              -0.101
APQ_P_APQ_P_CP                0.056
EHQ_EHQ_Total                 0.036
APQ_P_APQ_P_PM                0.029
MRI_Track_Age_at_Scan         0.014
APQ_P_APQ_P_PP               -0.011
ColorVision_CV_Score         -0.006
dtype: float64


## Categorical metadata: distribution and association with ADHD_Outcome (chi-square-style via groupby rates)

In [5]:

cat_cols = [c for c in train_cat.columns if c != 'participant_id']
cat_full = train_cat.merge(train_sol[['participant_id','ADHD_Outcome']], on='participant_id')

for c in cat_cols:
    rate = cat_full.groupby(c)['ADHD_Outcome'].agg(['mean','count'])
    spread = rate['mean'].max() - rate['mean'].min()
    print(f'{c}: ADHD-rate spread across categories = {spread:.3f}, n_categories={rate.shape[0]}, missing={train_cat[c].isnull().sum()}')


Basic_Demos_Enroll_Year: ADHD-rate spread across categories = 0.497, n_categories=6, missing=0
Basic_Demos_Study_Site: ADHD-rate spread across categories = 0.172, n_categories=4, missing=0
PreInt_Demos_Fam_Child_Ethnicity: ADHD-rate spread across categories = 0.060, n_categories=4, missing=43
PreInt_Demos_Fam_Child_Race: ADHD-rate spread across categories = 0.500, n_categories=10, missing=54
MRI_Track_Scan_Location: ADHD-rate spread across categories = 0.233, n_categories=4, missing=3
Barratt_Barratt_P1_Edu: ADHD-rate spread across categories = 0.200, n_categories=7, missing=15
Barratt_Barratt_P1_Occ: ADHD-rate spread across categories = 0.253, n_categories=10, missing=31
Barratt_Barratt_P2_Edu: ADHD-rate spread across categories = 0.105, n_categories=7, missing=198
Barratt_Barratt_P2_Occ: ADHD-rate spread across categories = 0.121, n_categories=10, missing=222


## Connectome: summary statistics, variance, PCA explained variance

In [6]:

connectome_cols = [c for c in train_fcm.columns if c != 'participant_id']
X_fcm = train_fcm[connectome_cols].astype('float32')

edge_var = X_fcm.var().sort_values(ascending=False)
print('Top 10 highest-variance edges:')
print(edge_var.head(10))
print()
print('Variance distribution summary:')
print(edge_var.describe())


Top 10 highest-variance edges:
159throw_177thcolumn    0.123547
80throw_177thcolumn     0.103169
70throw_80thcolumn      0.095602
70throw_159thcolumn     0.094143
80throw_151thcolumn     0.086971
87throw_151thcolumn     0.085667
80throw_81thcolumn      0.084562
159throw_186thcolumn    0.083153
151throw_188thcolumn    0.082862
151throw_159thcolumn    0.081325
dtype: float32

Variance distribution summary:
count    19900.000000
mean         0.035975
std          0.008079
min          0.006368
25%          0.030193
50%          0.034409
75%          0.040416
max          0.123547
dtype: float64


In [7]:

# Redundancy check: how correlated are the connectome edges with each other (sampled, since 19900x19900 is too large)
rng = np.random.default_rng(42)
sample_cols = rng.choice(connectome_cols, size=1500, replace=False)
corr_sub = X_fcm[sample_cols].corr()
upper = corr_sub.where(np.triu(np.ones(corr_sub.shape), k=1).astype(bool))
high_corr_frac = (upper.abs() > 0.7).sum().sum() / upper.notna().sum().sum()
print(f'Fraction of edge-pairs (sampled 1500 edges) with |corr| > 0.7: {high_corr_frac:.4f}')
print('-> indicates substantial redundancy in the connectome, justifying dimensionality reduction / feature selection')


Fraction of edge-pairs (sampled 1500 edges) with |corr| > 0.7: 0.0001
-> indicates substantial redundancy in the connectome, justifying dimensionality reduction / feature selection


In [8]:

# PCA explained variance on the full connectome (fit on all 1213 train subjects for EDA purposes only;
# the leakage-safe version is refit on the train split alone in notebook 03)
scaler_eda = StandardScaler(with_mean=True, with_std=False)  # connectome already bounded [-1,1]; center only
pca_eda = PCA(n_components=0.95, svd_solver='full', random_state=123)
X_pca_eda = pca_eda.fit_transform(X_fcm.values)
print('N components for 95% variance:', X_pca_eda.shape[1])
cum_var = np.cumsum(pca_eda.explained_variance_ratio_)
for target in [0.5, 0.7, 0.8, 0.9, 0.95]:
    n_comp = np.searchsorted(cum_var, target) + 1
    print(f'{int(target*100)}% variance explained by {n_comp} components')


N components for 95% variance: 902
50% variance explained by 108 components
70% variance explained by 292 components
80% variance explained by 453 components
90% variance explained by 708 components
95% variance explained by 902 components


## Connectome edges most associated with ADHD (univariate F-score / mutual information, whole-dataset for EDA only — NOT used for final feature selection, which is done inside CV in notebook 04)

In [9]:

y_aligned = train_fcm.merge(train_sol[['participant_id','ADHD_Outcome']], on='participant_id')['ADHD_Outcome'].values

f_scores, p_values = f_classif(X_fcm.values, y_aligned)
f_series = pd.Series(f_scores, index=connectome_cols).sort_values(ascending=False)
print('Top 15 connectome edges by ANOVA F-score vs ADHD_Outcome:')
print(f_series.head(15))
print()
print('Number of edges with p < 0.01:', int((p_values < 0.01).sum()), '/', len(p_values))
print('Number of edges with p < 0.05:', int((p_values < 0.05).sum()), '/', len(p_values))


Top 15 connectome edges by ANOVA F-score vs ADHD_Outcome:
166throw_184thcolumn    19.593731
78throw_170thcolumn     18.704387
1throw_16thcolumn       18.364494
2throw_166thcolumn      17.912034
11throw_166thcolumn     17.341323
78throw_189thcolumn     17.232210
58throw_163thcolumn     16.250615
76throw_170thcolumn     15.640209
105throw_166thcolumn    15.300160
53throw_127thcolumn     15.220866
50throw_197thcolumn     15.002168
159throw_166thcolumn    14.826407
0throw_166thcolumn      14.700486
50throw_189thcolumn     14.478515
34throw_130thcolumn     14.278616
dtype: float64

Number of edges with p < 0.01: 369 / 19900
Number of edges with p < 0.05: 1374 / 19900


## Where is the strongest predictive signal?

In [10]:

print('EDA SUMMARY')
print('='*60)
print(f'- ADHD class imbalance: {y.value_counts()[1]}/{y.value_counts()[0]} (~2.2:1), moderate imbalance')
print(f'- Strongest quantitative metadata correlate with ADHD_Outcome: {corr_series.index[0]} (rho={corr_series.iloc[0]:.3f})')
print(f'- Top 5 metadata correlations are all SDQ (behavioral questionnaire) subscales -- expected,')
print(f'  since SDQ Hyperactivity/Conduct/Difficulties are clinically adjacent to ADHD symptomatology.')
print(f'- Connectome: {int((p_values < 0.05).sum())}/{len(p_values)} edges individually significant at p<0.05 (univariate, whole-data),')
print(f'  consistent with a diffuse, high-dimensional but individually weak neuroimaging signal.')
print(f'- Connectome has heavy redundancy ({high_corr_frac:.1%} of sampled edge pairs |corr|>0.7), PCA compresses to')
print(f'  {X_pca_eda.shape[1]} components at 95% variance -- supports dimensionality reduction.')
print()
print('CONCLUSION: Strongest signal source = METADATA (SDQ behavioral subscales) + CONNECTOME (diffuse, high-dim).')
print('Both sources carry real signal; representation comparison in notebook 04 will quantify CONNECTOME vs')
print('CONNECTOME+METADATA vs METADATA-only using proper cross-validation.')


EDA SUMMARY
- ADHD class imbalance: 831/382 (~2.2:1), moderate imbalance
- Strongest quantitative metadata correlate with ADHD_Outcome: SDQ_SDQ_Hyperactivity (rho=0.548)
- Top 5 metadata correlations are all SDQ (behavioral questionnaire) subscales -- expected,
  since SDQ Hyperactivity/Conduct/Difficulties are clinically adjacent to ADHD symptomatology.
- Connectome: 1374/19900 edges individually significant at p<0.05 (univariate, whole-data),
  consistent with a diffuse, high-dimensional but individually weak neuroimaging signal.
- Connectome has heavy redundancy (0.0% of sampled edge pairs |corr|>0.7), PCA compresses to
  902 components at 95% variance -- supports dimensionality reduction.

CONCLUSION: Strongest signal source = METADATA (SDQ behavioral subscales) + CONNECTOME (diffuse, high-dim).
Both sources carry real signal; representation comparison in notebook 04 will quantify CONNECTOME vs
CONNECTOME+METADATA vs METADATA-only using proper cross-validation.


In [11]:

os.makedirs('reports', exist_ok=True)
eda_summary = pd.DataFrame({
    'quantitative_feature_corr_with_ADHD': corr_series
}).reset_index().rename(columns={'index':'feature'})
eda_summary.to_csv('reports/eda_summary.csv', index=False)
eda_summary.head(20)


,feature,quantitative_feature_corr_with_ADHD
0,SDQ_SDQ_Hyperactivity,0.548421
1,SDQ_SDQ_Externalizing,0.518822
2,SDQ_SDQ_Difficulties_Total,0.475942
3,SDQ_SDQ_Generating_Impact,0.417394
4,SDQ_SDQ_Conduct_Problems,0.283128
5,SDQ_SDQ_Internalizing,0.274974
6,SDQ_SDQ_Peer_Problems,0.229329
7,SDQ_SDQ_Emotional_Problems,0.216942
8,SDQ_SDQ_Prosocial,-0.198440
9,APQ_P_APQ_P_OPD,0.143217
